<center><h1>Breast Cancer (<em>TCGA-BRCA</em>) Subtype Classification</h1></center></br>

</br><h4>Data Preprocessing:</h4>

In [1]:
%run 1_Preprocess_TCGA_BRCA.ipynb
preprocessed_df = getPreprocessedData()

Original TCGA-BRCA Dataset Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 705 entries, 0 to 704
Columns: 1941 entries, rs_CLEC3A to histological.type
dtypes: float64(827), int64(1110), object(4)
memory usage: 10.4+ MB

No of Missing Values-Feature Dataset: 389

 Missing values of Feature Dataset:
 rs_CLEC3A              0
rs_CPB1                0
rs_SCGB2A2             0
rs_SCGB1D2             0
rs_TFF1                0
                    ... 
vital.status           0
PR.Status            122
ER.Status            122
HER2.Final.Status    145
histological.type      0
Length: 1941, dtype: int64

No of Missing Values - Feature Dataset: 0 

<class 'pandas.core.frame.DataFrame'>
Index: 560 entries, 0 to 649
Columns: 1941 entries, rs_CLEC3A to histological.type
dtypes: float64(827), int64(1110), object(4)
memory usage: 8.3+ MB
Unique values in ER.Status:
['Positive' 'Negative' 'Performed but Not Available' 'Indeterminate'
 'Not Performed'] 

Unique values in PR.Status:
['Positive'

</br><h4>Hetrograph Construction:</h4>

In [2]:
%run 2_TCGA_BRCA_Heterograph_Construction.ipynb
hetrograph = construct_hetrograph(preprocessed_df, top_k = None)

C:\Users\dular\.conda\envs\meditech_env\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
C:\Users\dular\.conda\envs\meditech_env\lib\site-packages\torch_geometric\typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


Creating deterministic biological mappings...
Creating deterministic subtype mappings...

Mapping summary:
    gene: 604 nodes
    protein: 223 nodes
    cnv: 860 nodes
    mutation: 249 nodes
    subtype: 5 nodes
    case: 512 nodes

Saving biological mappings to files...
    Main mappings saved to: graph_mappings\biological_mappings.json
    Individual mappings saved to: graph_mappings/
    Reverse mappings saved to: graph_mappings\reverse_biological_mappings.json

Defining Case and Subtype Nodes...
Defining Multi-Omic Nodes...

Defining Multi-Omics - Pateint Relations...
	Defining "Gene in Case" Metapath...
	Defining "Protein in Case" Metapath...
	Defining "CNV in Case" Metapath...
	Defining "Mutation in Case" Metapath...
	Defining "Case has Gene" Metapath...
	Defining "Case has Protein" Metapath...
	Defining "Case has CNV" Metapath...
	Defining "Case has Mutation" Metapath...

Defining Patient - Patient Similarity Edges...
	Defining "Case Similar to Case" Metapath...
	Defining "Cas

</br><h4>Hetrograph 2D Visualization:</h4>

In [3]:
# %run 3_Hetrograph_Visualization.ipynb
# hetrograph_2d_visualization(hetrograph)

</br><h4>Hetrograph 3D Visualization:</h4>

In [4]:
# %run 3_Hetrograph_Visualization.ipynb
# hetrograph_3d_visualization(hetrograph)

</br><h4>Training HAN Model:</h4>

In [16]:
%run ./HAN_Model/HAN_Model.ipynb
target_node_type='case'
max_path_length=5
max_paths_per_length = 30
max_metapath_combinations=75
min_paths_per_set=3
max_paths_per_set=6

results = run_comprehensive_metapath_experiment(hetrograph, target_node_type, max_path_length,\
                                                max_paths_per_length, max_metapath_combinations, min_paths_per_set, max_paths_per_set)

Starting comprehensive metapath experiment...

This will test multiple metapath combinations including all relations.

STARTING ENHANCED DYNAMIC META-PATH EXPERIMENT

Graph Structure:
  Node types: ['case', 'subtype', 'gene', 'protein', 'cnv', 'mutation']

  Edge types: [('gene', 'in', 'case'), ('protein', 'in', 'case'), ('cnv', 'in', 'case'), ('mutation', 'in', 'case'), ('case', 'has', 'gene'), ('case', 'has', 'protein'), ('case', 'has', 'cnv'), ('case', 'has', 'mutation'), ('case', 'similar_to', 'case'), ('case', 'coexpr_with', 'case'), ('gene', 'interacts', 'gene'), ('protein', 'ppi', 'protein'), ('cnv', 'co_cnv', 'cnv'), ('mutation', 'co_mutation', 'mutation'), ('gene', 'encodes', 'protein'), ('mutation', 'in_gene', 'gene'), ('cnv', 'affects', 'gene'), ('case', 'has_subtype', 'subtype'), ('subtype', 'characterized_by', 'gene'), ('subtype', 'characterized_by', 'protein'), ('subtype', 'characterized_by', 'cnv'), ('subtype', 'characterized_by', 'mutation')]

Node Count:
  case: 512 no

In [17]:
get_available_metapath_rankings('metapath_logs/metapath_results.csv', top_n=10)


Top 10 metapath sets from metapath_logs/metapath_results.csv:
Rank   Test Acc   Val Acc    Train Acc  #Paths   Description                                                           
1      0.7532     0.7013     0.4721     6        case--has-->gene -> gene--interacts-->gene -> gene--in-->case ...     
2      0.7532     0.7532     0.6620     5        case--similar_to-->case | case--has_subtype-->subtype -> subty...     
3      0.7532     0.7532     0.6983     6        case--has_subtype-->subtype -> subtype--characterized_by-->gen...     
4      0.7403     0.7273     0.6620     6        case--similar_to-->case | case--coexpr_with-->case | case--has...     
5      0.7403     0.7273     0.6620     5        case--similar_to-->case | case--coexpr_with-->case | case--has...     
6      0.7403     0.7403     0.7039     5        case--has_subtype-->subtype -> subtype--characterized_by-->gen...     
7      0.7403     0.7273     0.6453     4        case--similar_to-->case | case--has_subtype-->su

In [18]:
final_results = train_final_model_from_csv(
        hetero_data=hetrograph,
        csv_file_path='metapath_logs/metapath_results.csv',
        rank_number=1,
        model_save_path='models/final_hetegat_rank1.ckpt',
        target_node_type='case',
        nb_epochs=300,
        patience=100,
        split_ratios=(0.7, 0.15, 0.15)
    )


TRAINING FINAL MODEL FROM CSV RESULTS

Loading metapath set from rank 1:
  
Test accuracy: 0.7532
  
Description: case--has-->gene -> gene--interacts-->gene -> gene--in-->case | case--has-->gene -> gene--encodes-->protein -> protein--in-->case | case--has-->protein -> protein--ppi-->protein -> protein--in-->case | case--has-->cnv -> cnv--co_cnv-->cnv -> cnv--in-->case | case--has-->cnv -> cnv--affects-->gene -> gene--in-->case | case--has-->mutation -> mutation--co_mutation-->mutation -> mutation--in-->case
 
 Parsed 6 metapaths:
    1. [('case', 'has', 'gene'), ('gene', 'interacts', 'gene'), ('gene', 'in', 'case')]
    2. [('case', 'has', 'gene'), ('gene', 'encodes', 'protein'), ('protein', 'in', 'case')]
    3. [('case', 'has', 'protein'), ('protein', 'ppi', 'protein'), ('protein', 'in', 'case')]
    4. [('case', 'has', 'cnv'), ('cnv', 'co_cnv', 'cnv'), ('cnv', 'in', 'case')]
    5. [('case', 'has', 'cnv'), ('cnv', 'affects', 'gene'), ('gene', 'in', 'case')]
    6. [('case', 'has', 

In [19]:
%run 4_Multi_Omic_Contribution_Analysis.ipynb
csv_file = multi_omic_contribution_analysis(
    hetrograph, 
    model_path='models/final_hetegat_rank1.ckpt',
    metapath_csv='metapath_logs/metapath_results.csv',
    rank_number=1,
    mappings_dir='graph_mappings',
    output_file='multi_omic_contribution_analysis.csv',
    top_k= None
)

Extracting Biomarkers With Persistent Mappings

Using metapath set from rank 1:
  Test accuracy: 0.7532
  Parsed 6 metapaths for analysis

Loading persistent biological mappings from files...
Persistent biological mappings loaded successfully!

Loaded mappings:
  gene: 604 features
  protein: 223 features
  cnv: 860 features
  mutation: 249 features
  subtype: 5 features
  case: 512 features

Created feature-to-omic mapping for 1936 features using persistent mappings

Analyzing 6 metapaths for biological significance...
  Metapath 0: case -> --has-->gene -> --interacts-->gene -> --in-->case (weight: 1.76)
  Metapath 1: case -> --has-->gene -> --encodes-->protein -> --in-->case (weight: 1.76)
  Metapath 2: case -> --has-->protein -> --ppi-->protein -> --in-->case (weight: 1.76)
  Metapath 3: case -> --has-->cnv -> --co_cnv-->cnv -> --in-->case (weight: 1.41)
  Metapath 4: case -> --has-->cnv -> --affects-->gene -> --in-->case (weight: 1.41)
  Metapath 5: case -> --has-->mutation -> --co

In [20]:
%run 5_Cell_Line_Mapping.ipynb
contributed_omic_csv="omic_contribution_analysis/multi_omic_contribution_analysis.csv"
run_filtered_extraction(contributed_omic_csv) 
# run_complete_extraction()

Filtered Multi-Omic Data Extraction

 Loading target features from contribution analysis...
  Loaded EXACT target features:
   Gene features: 604
   Protein features: 223
   CNV features: 860
   Mutation features: 249

 Creating Ensembl ID → Gene Symbol mapping...
  Created mapping for 112404 Ensembl IDs

 Table 1: Genetic Features (Mutations) - Filtered
------------------------------------------------------------
 Filtered 111/525 mutation features
 Found 111 mappings across 49 cell lines
 
Sample:
Cell_Line Genetic_Feature
    OCUBM        MLL3_mut
    OCUBM      PIK3CA_mut
    OCUBM        TP53_mut
     T47D      ARID1A_mut
     T47D      PIK3CA_mut

 Table 2: Genetic Features (Mutations) - Filtered
------------------------------------------------------------
 Filtered 111/525 mutation features
 Found 111 mappings across 49 cell lines
 
Sample:
Cell_Line Genetic_Feature
    OCUBM        MLL3_mut
    OCUBM      PIK3CA_mut
    OCUBM        TP53_mut
     T47D      ARID1A_mut
     T47D 

In [21]:
%run 5_Cell_Line_Mapping.ipynb
cell_line_table_dir = 'complete_breast_multiomics_all_features'
results = run_enhanced_mapping(cell_line_table_dir)

Enhanced Multi-Omic Contribution Analysis + Cell Line Mapping
 
Loading multi-omic contribution analysis...
 Loaded 9680 contribution records
 Cancer subtypes: ['Basal-Like' 'HER2-Enriched' 'Luminal-A' 'Luminal-B' 'Undefined']
 Omic types: ['cnv' 'gene' 'mutation' 'protein']

 Loading extracted cell line tables...
  Loaded genetic_1: 525 mappings
  Loaded genetic_2: 525 mappings
  Loaded protein: 10058 mappings
  Loaded gene_counts: 1587662 mappings
  Loaded gene_rpkm: 1587662 mappings
  Loaded gene_tpm: 1361687 mappings
  Loaded cnv_matrix: 829794 mappings

 Mapping features to cell lines...
  Processing 0/9680 features...
  Processing 1000/9680 features...
  Processing 2000/9680 features...
  Processing 3000/9680 features...
  Processing 4000/9680 features...
  Processing 5000/9680 features...
  Processing 6000/9680 features...
  Processing 7000/9680 features...
  Processing 8000/9680 features...
  Processing 9000/9680 features...

 Enhanced 9680 contribution records with cell line m